In [1]:
import toollib as tl
from pathlib import Path
import subprocess
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

# 配置信息部分

In [3]:
# 数据库账号和密码
user,passwd = '',''

# 订单回溯数据保存路径
req_dir = "req_dir" # 如果不冲突可以不用改

# 设置国家信息
country_abbr = 'th'

# featurelib服务配置
ser_workers = 10
port = 10010
python_bin='~/miniconda3/envs/py310/bin/' # miniconda中 py310环境的bin目录
featurelib_path='~/featurelib/' # 自己环境中的featurelib代码路径

# 需要跑的特征模块注意服务名称的对应关系
fea_mappings={
    "loan_th":['loan_th_base_v1','loan_th_base_v2','loan_th_base_v3_2'],
    "sms_th":['sms_th_base_v1','sms_th_comp_v1','sms_th_loan0_v1','sms_th_cate_v1'],
    "app_th":['app_th_base_v1','app_th_comp_v1','app_th_gcate_v1','app_th_comp0_v1','app_th_cate_v1'],   
    "call_th":['app_th_base_v1','call_th_base0_v1'],   
    "cross_th":['cross_th_base_v1'],
    "dev_th":['dev_tz_base_v1','dev_tz_base_v2'],
    "ui_th":['ui_th_base_v2']
}

# 根据自己的时间要求设置待取样的放款时间
loan_df = tl.get_loans(country_abbr,user,passwd,start_date='2024-08-01',end_date= '2024-08-06')

sql:

    with app_base_info as (
        select
         a.app_order_id,a.id_card_number,a.user_name,a.phone_number,a.device_type,a.apply_time,a.product_info,
         a.personal_info,a.work_info,a.contact_info,a.bank_account_info,a.ktp_info,a.face_recognition_info,cr.term,
         cr.days_per_term,u.source,row_number() over (partition by a.app_order_id order by u.create_time desc) rn,r.tx_id
        from ath_system.t_app_order a
        left join ath_system.t_contract cr on cr.app_order_id= a.app_order_id
        left  join ath_system.t_risk_req_record r on a.app_order_id = r.biz_id
        left join ath_system.t_app_user u on a.acq_channel = u.acq_channel and a.user_id = u.user_id  and a.apply_time >= u.create_time
    )select a.*,b.id_card_number,b.phone_number,b.user_name,b.device_type,b.apply_time,b.personal_info,b.work_info,b.contact_info,b.bank_account_info,
            b.ktp_info,b.face_recognition_info,b.source,b.term,b.days_per_term,c.acq_channel_name,b.tx_id
            fr

In [3]:
# todo 建议在此进行自己的分析相关操作
# 根据自己的条件设置采样的
sample_df = loan_df[(loan_df['new_old_user_status']>0)&(loan_df['extension_count']==0)&(loan_df['agr_pd7']==1)]
sample_df['bad'] = sample_df['def_pd7']

print(sample_df.shape,sample_df['app_order_id'].nunique())

(2915, 98) 2915


# 样本信息持久化

In [4]:
# 按照放款天对数据进行切块
sample_df['file_name'] = 'raw_df_' + sample_df['loan_time'].dt.strftime("%Y-%m-%d") + '.parquet'
sample_df.to_parquet('sample_df.parquet',compression='zstd')

# 下载数据

In [5]:
req_dir = Path(req_dir)
req_dir.mkdir(exist_ok=True)
sample_df = pd.read_parquet('sample_df.parquet')
print(sample_df.shape,sample_df['app_order_id'].nunique())

(2915, 99) 2915


In [8]:
file_names = sorted(sample_df['file_name'].unique())
for fname in file_names:
    orderlist = sample_df[sample_df['file_name']==fname]['app_order_id'].to_list()
    print(fname,len(orderlist),'开始下载：=================>')
    req_df = tl.get_sample_req(orderlist,'th',user,passwd)
    req_df.to_parquet(req_dir/fname,compression='zstd')

raw_df_2024-08-01.parquet 666 开始下载：=================>
共计输入666条数据，清理重复数据后共666条数据
成功查询到进件相关信息,共计(666, 28),对应的用户数量388
剔除异常订单后，还剩余666订单，对应388个用户
基于user_id关联的放款单,共计(4678, 34)
基于user_id关联的进件订单,共计(7198, 33)
基于user_id关联的合同单,共计(4499, 40)
基于user_id关联设备数据,共计(17100, 4)


100%|██████████| 666/666 [00:00<00:00, 37315.07it/s]


开始组装数据,共计666条数据


666it [00:59, 11.28it/s]


raw_df_2024-08-02.parquet 595 开始下载：=================>
共计输入595条数据，清理重复数据后共595条数据
成功查询到进件相关信息,共计(595, 28),对应的用户数量327
剔除异常订单后，还剩余595订单，对应327个用户
基于user_id关联的放款单,共计(4101, 34)
基于user_id关联的进件订单,共计(6623, 33)
基于user_id关联的合同单,共计(3921, 40)
基于user_id关联设备数据,共计(13640, 4)


...

569it [00:55, 10.23it/s]


raw_df_2024-08-05.parquet 540 开始下载：=================>
共计输入540条数据，清理重复数据后共540条数据
成功查询到进件相关信息,共计(540, 28),对应的用户数量316
剔除异常订单后，还剩余540订单，对应316个用户
基于user_id关联的放款单,共计(4163, 34)
基于user_id关联的进件订单,共计(6111, 33)
基于user_id关联的合同单,共计(3842, 40)
基于user_id关联设备数据,共计(12801, 4)


100%|██████████| 540/540 [00:00<00:00, 37336.17it/s]


开始组装数据,共计540条数据


540it [00:51, 10.42it/s]


# 开始计算特征

In [11]:
# 下载好的数据所在的目录
req_files = tl.data_of_dir(req_dir)
print(len(req_files))
req_files[0:2]

5


['req_dir/raw_df_2024-08-01.parquet', 'req_dir/raw_df_2024-08-02.parquet']

In [ ]:
uri = f'http://127.0.0.1:{port}'
client_workers = ser_workers+5

# 回溯特征:
for ser_name,model_names in fea_mappings.items():
    tl.start_ser(ser_name,port=port,workers=ser_workers,python_bin=python_bin,featurelib_path=featurelib_path)
    for model_name in model_names:
        tl.batch_request_featurelib(req_files,model_name,uri=uri,max_workers=client_workers)


longxia+ 43119     1  1 03:42 ?        00:00:00 /home/longxiaolei/miniconda3/envs/py310/bin/python /home/longxiaolei/miniconda3/envs/py310/bin/gunicorn -w 10 --threads 1 --chdir feature_service/loan_th -D loan_th_service:app -b 0.0.0.0:10010 --access-logfile=logs/access.log --error-logfile=logs/error.log
longxia+ 43120 43119 99 03:42 ?        00:00:03 /home/longxiaolei/miniconda3/envs/py310/bin/python /home/longxiaolei/miniconda3/envs/py310/bin/gunicorn -w 10 --threads 1 --chdir feature_service/loan_th -D loan_th_service:app -b 0.0.0.0:10010 --access-logfile=logs/access.log --error-logfile=logs/error.log
longxia+ 43121 43119 99 03:42 ?        00:00:03 /home/longxiaolei/miniconda3/envs/py310/bin/python /home/longxiaolei/miniconda3/envs/py310/bin/gunicorn -w 10 --threads 1 --chdir feature_service/loan_th -D loan_th_service:app -b 0.0.0.0:10010 --access-logfile=logs/access.log --error-logfile=logs/error.log
longxia+ 43122 43119 99 03:42 ?        00:00:03 /home/longxiaolei/miniconda3/envs/

loan_th_base_v3_2:  60%|██████    | 3/5 [15:40<10:09, 304.92s/it]

## 将跑完得特征分块合并并落库

In [7]:
for k,v in fea_mappings.items():
    for model_name in v:
        fea_files = tl.data_of_dir(model_name)
        feature_data_df = tl.batch_load_data(fea_files)
        feature_data_df.to_parquet(f"{model_name}.parquet",compression='zstd')
        print(f"{model_name}完成加载,{feature_data_df.shape}")

100%|██████████| 5/5 [00:00<00:00,  5.17it/s]


loan_th_base_v1完成加载,(2915, 2810)


100%|██████████| 5/5 [00:02<00:00,  2.10it/s]


loan_th_base_v2完成加载,(2915, 7388)


100%|██████████| 4/4 [00:05<00:00,  1.42s/it]


loan_th_base_v3_2完成加载,(2375, 16185)


In [ ]:
"""
jupyter nbconvert --to script get_sample_req_features.ipynb

conda activate py310

nohup python get_sample_req_features.py &
"""